# Round 21A — Recreate the Public 3rd-Place Strategy

This is the **canonical executable reproduction notebook**.

It does not copy another notebook. It independently rebuilds the strategy publicly described by the 3rd-place MITSUI solution:

- one modeling problem per target;
- target-specific routing from `target_pairs.csv`;
- lag, rolling, difference, and safe-log feature families;
- median imputation and standardization;
- LightGBM + Random Forest + XGBoost base learners;
- XGBoost level-1 stacker.

Two stacks are evaluated:

1. **`source_stack`** — recreates the publicly described in-sample stacking structure.
2. **`causal_oof_stack`** — our research-grade correction using chronological OOF meta-features.

The public writeup describes negative lags but does not unambiguously define a causal implementation. The codebase preserves a forensic literal-shift implementation for reproducibility research, but this notebook **does not train or promote future-looking features**.

The historical private leaderboard score (0.60229) is not directly comparable to our offline development folds. This notebook evaluates only the project's established chronological development protocol and does not touch the final reserved origins.


In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import subprocess
import time

import boto3
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path("/home/sagemaker-user/projects/commodity-prediction-current")
SRC = ROOT / "src"
CONFIG_PATH = ROOT / "configs/third_place_reproduction.json"
ARTIFACT_ROOT = ROOT / "artifacts/third_place_reproduction"
BASELINE_CACHE = ROOT / "artifacts/baselines/current_market"

import sys
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from commodity_prediction.data import load_data, make_folds, reconstruct_targets
from commodity_prediction.metrics import (
    score,
    daily_rank_correlations,
    paired_block_interval,
)
from commodity_prediction.competitive.third_place_reproduction import (
    build_causal_pair_features,
    build_forensic_source_described_features,
    choose_stratified_targets,
    fit_causal_oof_stack,
    fit_source_described_stack,
    prefix_invariant_feature_check,
    prepare_train_valid,
)

cfg = json.loads(CONFIG_PATH.read_text())
MODE = os.environ.get("ROUND21A_MODE", "smoke").strip().lower()
assert MODE in {"smoke", "panel", "full"}, MODE

mode_to_n = {
    "smoke": cfg["smoke_targets"],
    "panel": cfg["panel_targets"],
    "full": 424,
}

PYTHON = str(ROOT / ".venv/bin/python")
HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
BRANCH = subprocess.check_output(["git", "branch", "--show-current"], cwd=ROOT, text=True).strip()

print("PROJECT_ROOT:", ROOT)
print("MODE:", MODE)
print("GIT_HEAD:", HEAD)
print("GIT_BRANCH:", BRANCH)
print("PYTHON:", PYTHON)
print("CONFIG:", CONFIG_PATH)


PROJECT_ROOT: /home/sagemaker-user/projects/commodity-prediction-current
MODE: smoke
GIT_HEAD: eb44b0fdef45228d9c1c5d230b13d58e417fa39d
GIT_BRANCH: feat/third-place-reproduction
PYTHON: /home/sagemaker-user/projects/commodity-prediction-current/.venv/bin/python
CONFIG: /home/sagemaker-user/projects/commodity-prediction-current/configs/third_place_reproduction.json


## 1. Environment and model-library provenance

In [2]:
import sklearn
import lightgbm
import xgboost

versions = {
    "python": sys.version.split()[0],
    "scikit_learn": sklearn.__version__,
    "lightgbm": lightgbm.__version__,
    "xgboost": xgboost.__version__,
}
versions


{'python': '3.12.14',
 'scikit_learn': '1.9.0',
 'lightgbm': '4.7.0',
 'xgboost': '3.4.1'}

## 2. Load official raw data and reproduce the chronological fold

In [3]:
x, y, pairs = load_data(ROOT)

folds, development_stop = make_folds(
    len(x),
    {
        "holdout_dates": 252,
        "validation_dates": 180,
        "n_folds": 3,
        "purge_dates": 5,
        "min_train_dates": 600,
    },
)
fold = folds[cfg["fold_number"]]

selected_targets = choose_stratified_targets(pairs, mode_to_n[MODE])
pair_map = pairs.set_index("target")["pair"].to_dict()

print("x shape:", x.shape)
print("y shape:", y.shape)
print("target_pairs shape:", pairs.shape)
print("fold:", fold)
print("development_stop:", development_stop)
print("selected_target_count:", len(selected_targets))
print("selected_targets:", selected_targets)


x shape: (1961, 557)
y shape: (1961, 424)
target_pairs shape: (424, 3)
fold: Fold(number=0, train_stop=1164, validation_start=1169, validation_stop=1349)
development_stop: 1709
selected_target_count: 8
selected_targets: ['target_0', 'target_1', 'target_106', 'target_107', 'target_212', 'target_213', 'target_318', 'target_319']


## 3. Correctness gates before fitting

In [4]:
reconstructed = reconstruct_targets(x, pairs)
mask = reconstructed.notna() & y.notna()
reconstruction_error = (reconstructed - y).where(mask).stack().abs()
max_reconstruction_error = float(reconstruction_error.max())
reconstruction_error_quantiles = {
    "p50": float(reconstruction_error.quantile(0.50)),
    "p95": float(reconstruction_error.quantile(0.95)),
    "p99": float(reconstruction_error.quantile(0.99)),
    "p999": float(reconstruction_error.quantile(0.999)),
    "max": max_reconstruction_error,
}
reconstruction_tolerance = float(cfg["target_reconstruction_tolerance"])

sample_pair = pairs.iloc[0]["pair"]
prefix_ok = prefix_invariant_feature_check(
    x,
    sample_pair,
    min(700, fold.train_stop),
    positive_lags=cfg["positive_lags"],
    rolling_windows=cfg["rolling_windows"],
)

print("target reconstruction error quantiles:", reconstruction_error_quantiles)
print("target reconstruction tolerance:", reconstruction_tolerance)
print("causal feature prefix invariance:", prefix_ok)

assert np.isfinite(max_reconstruction_error)
assert max_reconstruction_error <= reconstruction_tolerance
assert prefix_ok


target reconstruction error quantiles: {'p50': 4.0419056990259605e-16, 'p95': 1.2975731600306517e-15, 'p99': 1.6861512186494565e-15, 'p999': 2.1050869380978554e-15, 'max': 9.936084974879883e-07}
target reconstruction tolerance: 2e-06
causal feature prefix invariance: True


### Forensic negative-lag audit

The public writeup mentions positive and negative lags. A literal negative shift in a standard pandas matrix uses a future row. We preserve that behavior only as an auditable forensic reconstruction and prove here that it is **not prefix-invariant**. It is not used in the promotable training run.


In [5]:
forensic_pair = pairs.iloc[0]["pair"]
forensic_full = build_forensic_source_described_features(
    x.iloc[:80],
    forensic_pair,
    positive_lags=cfg["positive_lags"],
    negative_lags=cfg["forensic_negative_lags"],
    rolling_windows=cfg["rolling_windows"],
)
forensic_prefix = build_forensic_source_described_features(
    x.iloc[:50],
    forensic_pair,
    positive_lags=cfg["positive_lags"],
    negative_lags=cfg["forensic_negative_lags"],
    rolling_windows=cfg["rolling_windows"],
)

neg_cols = [c for c in forensic_full.columns if "__neg_lag" in c]
assert neg_cols, "Expected forensic negative-lag columns"
forensic_is_prefix_invariant = forensic_full.iloc[:50][neg_cols].equals(
    forensic_prefix[neg_cols]
)
print("forensic negative-lag prefix invariant:", forensic_is_prefix_invariant)
assert forensic_is_prefix_invariant is False


forensic negative-lag prefix invariant: False


## 4. Recreate the 3rd-place model strategy target by target

For every selected target, this cell visibly:

1. restricts the market inputs to its `target_pairs.csv` constituent(s);
2. rebuilds causal lag/rolling/difference/log feature families;
3. fits the **LightGBM, Random Forest, XGBoost** level-0 models;
4. fits a source-described **XGBoost** stacker from in-sample level-0 predictions;
5. fits our corrected chronological-OOF XGBoost stacker;
6. saves per-target predictions and receipts.

The public writeup does not disclose every exact hyperparameter or rolling-window length. Those reconstruction choices are explicitly stored in the canonical config rather than silently presented as source facts.


In [6]:
run_payload = json.dumps(
    {
        "git_head": HEAD,
        "mode": MODE,
        "fold": fold.number,
        "selected_targets": selected_targets,
        "config": cfg,
        "notebook": "notebooks/21_third_place_reproduction.ipynb",
    },
    sort_keys=True,
).encode()
run_id = hashlib.sha256(run_payload).hexdigest()[:16]
RUN_DIR = ARTIFACT_ROOT / run_id
TARGET_DIR = RUN_DIR / "targets"
TARGET_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "selected_targets.json").write_text(json.dumps(selected_targets, indent=2) + "\n")

results = []
started = time.time()

for i, target in enumerate(selected_targets, 1):
    pair = pair_map[target]
    stage = TARGET_DIR / target
    stage.mkdir(parents=True, exist_ok=True)
    result_path = stage / "result.json"
    prediction_path = stage / "predictions.parquet"

    if result_path.exists() and prediction_path.exists():
        result = json.loads(result_path.read_text())
        results.append(result)
        print(f"[{i}/{len(selected_targets)}] REUSED {target}")
        continue

    features = build_causal_pair_features(
        x,
        pair,
        positive_lags=cfg["positive_lags"],
        rolling_windows=cfg["rolling_windows"],
    )

    x_train = features.iloc[: fold.train_stop]
    x_valid = features.iloc[fold.validation_start : fold.validation_stop]
    y_train = y[target].iloc[: fold.train_stop]
    y_valid = y[target].iloc[fold.validation_start : fold.validation_stop]

    observed_train = y_train.notna()
    prepared = prepare_train_valid(x_train.loc[observed_train], x_valid)

    source_pred, source_parts = fit_source_described_stack(
        prepared.train,
        y_train.loc[observed_train].to_numpy(dtype=float),
        prepared.valid,
        seed=cfg["seed"],
        threads=cfg["threads"],
    )

    oof_pred, oof_parts = fit_causal_oof_stack(
        x_train,
        y_train,
        x_valid,
        seed=cfg["seed"],
        threads=cfg["threads"],
        n_splits=cfg["oof_splits"],
        gap=cfg["oof_gap"],
    )

    pred = pd.DataFrame(
        {
            "date_id": x_valid.index,
            "truth": y_valid.to_numpy(dtype=float),
            "source_stack": source_pred,
            "oof_stack": oof_pred,
            **{
                f"source_{name}": values
                for name, values in source_parts.items()
                if name != "stack"
            },
            **{
                f"oof_{name}": values
                for name, values in oof_parts.items()
                if name != "stack_oof"
            },
        }
    )
    pred.to_parquet(prediction_path, index=False)

    observed_valid = np.isfinite(pred["truth"])
    result = {
        "target": target,
        "pair": pair,
        "fold": fold.number,
        "feature_count": int(features.shape[1]),
        "train_rows": int(observed_train.sum()),
        "valid_rows": int(len(x_valid)),
        "source_stack_pearson": (
            float(
                np.corrcoef(
                    pred.loc[observed_valid, "truth"],
                    pred.loc[observed_valid, "source_stack"],
                )[0, 1]
            )
            if observed_valid.sum() > 2
            else None
        ),
        "oof_stack_pearson": (
            float(
                np.corrcoef(
                    pred.loc[observed_valid, "truth"],
                    pred.loc[observed_valid, "oof_stack"],
                )[0, 1]
            )
            if observed_valid.sum() > 2
            else None
        ),
        "completed_utc": datetime.now(timezone.utc).isoformat(),
    }
    result_path.write_text(json.dumps(result, indent=2, sort_keys=True) + "\n")
    results.append(result)

    elapsed = time.time() - started
    print(
        f"[{i}/{len(selected_targets)}] {target} | "
        f"features={features.shape[1]} | elapsed={elapsed:.1f}s"
    )

target_results = pd.DataFrame(results)
target_results


[1/8] target_0 | features=28 | elapsed=6.2s


[2/8] target_1 | features=77 | elapsed=18.2s


[3/8] target_106 | features=28 | elapsed=23.7s


[4/8] target_107 | features=77 | elapsed=34.6s


[5/8] target_212 | features=28 | elapsed=39.3s


[6/8] target_213 | features=77 | elapsed=49.8s


[7/8] target_318 | features=28 | elapsed=55.0s


[8/8] target_319 | features=77 | elapsed=66.0s


,target,pair,fold,feature_count,train_rows,valid_rows,source_stack_pearson,oof_stack_pearson,completed_utc
0,target_0,US_Stock_VT_adj_close,0,28,1088,180,-0.004963,-0.143805,2026-09-16T23:49:07.252894+00:00
1,target_1,LME_PB_Close - US_Stock_VT_adj_close,0,77,1061,180,0.032462,0.024719,2026-09-16T23:49:19.237152+00:00
2,target_106,US_Stock_VXUS_adj_close,0,28,1088,180,0.041624,-0.181296,2026-09-16T23:49:24.765502+00:00
3,target_107,LME_ZS_Close - US_Stock_VXUS_adj_close,0,77,1051,180,-0.052322,0.002294,2026-09-16T23:49:35.581027+00:00
4,target_212,FX_ZARUSD,0,28,1164,180,0.116738,0.017025,2026-09-16T23:49:40.305007+00:00
5,target_213,LME_PB_Close - FX_ZARUSD,0,77,1102,180,0.103158,0.161111,2026-09-16T23:49:50.800995+00:00
6,target_318,FX_NOKEUR,0,28,1164,180,-0.131062,-0.166601,2026-09-16T23:49:55.980075+00:00
7,target_319,LME_AH_Close - FX_NOKEUR,0,77,1102,180,0.085594,-0.007235,2026-09-16T23:50:07.015563+00:00


## 5. Replay the exact existing `current_market` baseline

In [7]:
BASELINE_CACHE.mkdir(parents=True, exist_ok=True)
baseline_path = BASELINE_CACHE / f"fold_{fold.number}.parquet"
baseline_key = (
    f"checkpoints/artifacts/{cfg['baseline_s3_lineage']}/"
    f"fold_{fold.number}/current_market/predictions.parquet"
)

if not baseline_path.exists():
    boto3.client("s3").download_file(
        cfg["baseline_s3_bucket"],
        baseline_key,
        str(baseline_path),
    )

baseline = pd.read_parquet(baseline_path)
if "date_id" in baseline.columns:
    baseline = baseline.set_index("date_id")

expected_targets = [f"target_{i}" for i in range(424)]
baseline = baseline[expected_targets]
y_valid_all = y.iloc[fold.validation_start : fold.validation_stop][expected_targets]

if not baseline.index.equals(y_valid_all.index):
    if len(baseline) == len(y_valid_all) and isinstance(baseline.index, pd.RangeIndex):
        baseline = baseline.copy()
        baseline.index = y_valid_all.index
    else:
        baseline = baseline.reindex(y_valid_all.index)

baseline_score = score(y_valid_all, baseline)
print("current_market fold score:", baseline_score)
assert abs(baseline_score - 0.40338108742296147) < 1e-10


current_market fold score: 0.40338108742296147


## 6. Replace only the recreated targets and score all 424 targets

Smoke/panel modes are cost gates only. Targets not yet recreated remain exactly equal to the preserved `current_market` predictions, so every reported official metric still spans all 424 targets.


In [8]:
def assemble_variant(prediction_column: str, weight: float) -> pd.DataFrame:
    out = baseline.copy()
    for target in selected_targets:
        p = pd.read_parquet(TARGET_DIR / target / "predictions.parquet").set_index("date_id")
        idx = out.index.intersection(p.index)
        out.loc[idx, target] = (
            (1.0 - weight) * out.loc[idx, target].to_numpy(dtype=float)
            + weight * p.loc[idx, prediction_column].to_numpy(dtype=float)
        )
    return out

scores = {"current_market": baseline_score}
correlations = {
    "current_market": daily_rank_correlations(y_valid_all, baseline)
}
predictions_by_variant = {"current_market": baseline}

for stack_name, prediction_column in [
    ("source_stack", "source_stack"),
    ("causal_oof_stack", "oof_stack"),
]:
    for weight in cfg["fixed_replacement_weights"]:
        name = f"{stack_name}__replace_{weight:.2f}"
        variant = assemble_variant(prediction_column, weight)
        predictions_by_variant[name] = variant
        scores[name] = score(y_valid_all, variant)
        correlations[name] = daily_rank_correlations(y_valid_all, variant)

score_table = (
    pd.Series(scores, name="official_metric")
    .sort_values(ascending=False)
    .rename_axis("variant")
    .reset_index()
)
score_table


,variant,official_metric
0,source_stack__replace_0.25,0.404442
1,source_stack__replace_0.50,0.403676
2,current_market,0.403381
3,source_stack__replace_0.75,0.402943
4,source_stack__replace_1.00,0.402272
5,causal_oof_stack__replace_0.25,0.399943
6,causal_oof_stack__replace_0.50,0.398205
7,causal_oof_stack__replace_0.75,0.397170
8,causal_oof_stack__replace_1.00,0.396784


In [9]:
fig = px.bar(
    score_table,
    x="variant",
    y="official_metric",
    title=f"Round 21A {MODE}: official 424-target metric",
    text_auto=".4f",
)
fig.update_layout(xaxis_tickangle=-35)
fig.show()


## 7. Per-target behavior and model diversity

In [10]:
fig = px.scatter(
    target_results,
    x="source_stack_pearson",
    y="oof_stack_pearson",
    hover_name="target",
    hover_data=["pair", "feature_count", "train_rows"],
    size="feature_count",
    title="Per-target Pearson correlation: source-described stack vs causal OOF stack",
)
fig.add_shape(
    type="line",
    x0=-1,
    y0=-1,
    x1=1,
    y1=1,
    line_dash="dash",
)
fig.show()


In [11]:
variant_corr = pd.DataFrame(
    {
        name: pred.to_numpy(dtype=float).ravel()
        for name, pred in predictions_by_variant.items()
    }
).corr()

fig = px.imshow(
    variant_corr,
    text_auto=".3f",
    aspect="auto",
    title="Prediction correlation across complete 424-target variants",
)
fig.show()


## 8. Paired block uncertainty relative to `current_market`

In [12]:
reference_corr = correlations["current_market"]
interval_rows = []

for name, corr in correlations.items():
    if name == "current_market":
        continue
    lo, hi = paired_block_interval(
        reference_corr,
        corr,
        repetitions=cfg["bootstrap_repetitions"],
        block=cfg["bootstrap_block_dates"],
        seed=cfg["seed"],
    )
    interval_rows.append(
        {
            "variant": name,
            "score": scores[name],
            "delta": scores[name] - baseline_score,
            "ci_lo": lo,
            "ci_hi": hi,
        }
    )

interval_table = pd.DataFrame(interval_rows).sort_values("delta", ascending=False)
interval_table


,variant,score,delta,ci_lo,ci_hi
0,source_stack__replace_0.25,0.404442,0.001061,-0.009702,0.012342
1,source_stack__replace_0.50,0.403676,0.000295,-0.012401,0.013747
2,source_stack__replace_0.75,0.402943,-0.000438,-0.014234,0.013487
3,source_stack__replace_1.00,0.402272,-0.001109,-0.015083,0.013363
4,causal_oof_stack__replace_0.25,0.399943,-0.003438,-0.010505,0.003079
5,causal_oof_stack__replace_0.50,0.398205,-0.005176,-0.015562,0.003739
6,causal_oof_stack__replace_0.75,0.397170,-0.006211,-0.017863,0.003985
7,causal_oof_stack__replace_1.00,0.396784,-0.006597,-0.018536,0.003809


In [13]:
if len(interval_table):
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=interval_table["delta"],
            y=interval_table["variant"],
            mode="markers",
            error_x=dict(
                type="data",
                symmetric=False,
                array=interval_table["ci_hi"] - interval_table["delta"],
                arrayminus=interval_table["delta"] - interval_table["ci_lo"],
            ),
        )
    )
    fig.add_vline(x=0.0, line_dash="dash")
    fig.update_layout(
        title="Official-metric delta vs current_market with paired block interval",
        xaxis_title="metric delta",
        yaxis_title="variant",
    )
    fig.show()


## 9. Persist the complete reproducibility receipt

In [14]:
best_variant = max(scores, key=scores.get)

summary = {
    "run_id": run_id,
    "mode": MODE,
    "notebook": "notebooks/21_third_place_reproduction.ipynb",
    "git_head": HEAD,
    "git_branch": BRANCH,
    "model_library_versions": versions,
    "fold": fold.number,
    "selected_targets": selected_targets,
    "selected_target_count": len(selected_targets),
    "scores": {k: float(v) for k, v in scores.items()},
    "baseline_score": float(baseline_score),
    "best_variant": best_variant,
    "best_score": float(scores[best_variant]),
    "paired_block_delta_intervals": {
        row["variant"]: [float(row["ci_lo"]), float(row["ci_hi"])]
        for row in interval_rows
    },
    "target_reconstruction_max_abs_error": max_reconstruction_error,
    "causal_prefix_invariance": bool(prefix_ok),
    "forensic_negative_lags_trained": False,
    "reserved_final_origins_evaluated": False,
    "source_reproduction_contract": {
        "source_described": [
            "one-model-per-target",
            "target_pairs routing",
            "lag features",
            "rolling mean/max features",
            "difference features",
            "safe log transform",
            "median imputation",
            "standardization",
            "LightGBM base learner",
            "Random Forest base learner",
            "XGBoost base learner",
            "XGBoost meta-model",
            "in-sample meta-predictions for source_stack",
        ],
        "source_unspecified_reconstruction_choices": cfg[
            "source_unspecified_reconstruction_choices"
        ],
    },
    "completed_utc": datetime.now(timezone.utc).isoformat(),
}

RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "summary.json").write_text(
    json.dumps(summary, indent=2, sort_keys=True) + "\n"
)
(RUN_DIR / "DONE.json").write_text(
    json.dumps(
        {
            "run_id": run_id,
            "status": "complete",
            "mode": MODE,
            "reserved_final_origins_evaluated": False,
            "completed_utc": datetime.now(timezone.utc).isoformat(),
        },
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

score_table.to_csv(RUN_DIR / "score_table.csv", index=False)
target_results.to_csv(RUN_DIR / "target_results.csv", index=False)
interval_table.to_csv(RUN_DIR / "paired_block_intervals.csv", index=False)

print("RUN_DIR:", RUN_DIR)
print("BEST_VARIANT:", best_variant)
print("BEST_SCORE:", scores[best_variant])
print("RESERVED_FINAL_ORIGINS_EVALUATED:", False)


RUN_DIR: /home/sagemaker-user/projects/commodity-prediction-current/artifacts/third_place_reproduction/e0027727b444f469
BEST_VARIANT: source_stack__replace_0.25
BEST_SCORE: 0.4044418534824003
RESERVED_FINAL_ORIGINS_EVALUATED: False


## Decision rule

Do **not** interpret this smoke run as exhaustion or as a leaderboard reconstruction.

Advance to the 64-target panel only if:

- the implementation passes all correctness gates;
- baseline parity is exact;
- predictions are finite;
- at least one recreated stack shows useful incremental signal or complementary error structure;
- no leakage or alignment defect is found.

The final reserved origins remain untouched until the full research program reaches its one-time final gate.
